# AU Case Study - Static Baseline + Correction Arms Overview (Read-Only)

> Data source: `011_static_baselines.py`; specification = `docs/B0_design_decisions.md`.
> This notebook is **read-only display only, and writes no artifacts**; all numbers come
> from saved files under `data/processed/static/` (nothing is hand-typed).
>
> **Arm matrix**: 14 arms = 2 static bases {Uniform, GPM} x {uncorrected, multiplicative
> N/P/NP, additive N/P/NP}; corrections **import the UK's shared module directly**
> (`shared_correction_utils`, zero changes to the numerical path); aggregation = two-stage
> Voronoi (frozen EPSG:3857 convention) -> 143 stations -> per-SA4 regional RMSE/MAE/Corr.
>
> **Deviations from the UK convention** (full list in `results_summary.json.deviations`):
> (1) region universe of 34/12 (corrected per the actual usable-station count); (2) proximity
> uses the pre-computed npz from the feature-assembly step (EPSG:7856) rather than the
> built-in EPSG:27700; (3) SA3 -> 'ITL3', demand_peak_mw -> 'Demand (MVA)' are pure alias
> adaptations; (4) demand = peak_mw (MW); (5) the PV sub-table is a pure statistical-layer
> station-set filter (allocation itself is not re-run).

In [ ]:
%matplotlib inline
# Environment and artifact paths (read-only)
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

AU_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STATIC = AU_DIR / "data" / "processed" / "static"

matrix = pd.read_csv(STATIC / "au_static_metrics.csv", encoding="utf-8-sig")
summary = json.loads((STATIC / "results_summary.json").read_text(encoding="utf-8"))
LOCS = summary["meta"]["region_order"]
ARMS = [a for a in matrix["arm"].drop_duplicates()]
print("Generated by:", summary["meta"]["generated_by"], "|", summary["meta"]["generated_at"])
print(f"Arms = {summary['meta']['n_arms']} | Regions = {summary['meta']['n_regions']} | "
      f"Usable stations = {summary['meta']['n_usable_stations']} | Rows = {len(matrix)}")
print("Demand convention:", summary["meta"]["demand_col"], "(primary convention)")
print("Randomness:", summary["meta"]["randomness"])

## 1. Main Table: 12-Region Mean per Arm (station_set = 'all')

The three UniAdd\* arms are **structurally degenerate arms** (alpha = base_std/offset_std;
within a Uniform-base region base_std = 0, so alpha = 0, so the arm is identical to Uni) --
the moment-matching definition is kept and reported as-is (matching the same behavior seen
in the UK static-baseline experiments).

In [ ]:
# Per-arm mean +/- std (ddof=0) pivot table (main table)
main = matrix[matrix["station_set"] == "all"].copy()
pivot = main.pivot_table(index="arm", columns="metric",
                         values=["mean_12regions", "std_ddof0_12regions"], sort=False)
tbl = pd.DataFrame({
    m: pivot[("mean_12regions", m)].map("{:.3f}".format) + " ± "
       + pivot[("std_ddof0_12regions", m)].map("{:.3f}".format)
    for m in ["rmse", "mae", "corr"]}).loc[ARMS]
tbl["structural degeneration"] = ["≡ Uni" if a in summary["uniform_additive_degeneration"]["affected_arms"]
                    else "" for a in tbl.index]
display(tbl)

# RMSE mean bar chart (colored by form)
meta = main.drop_duplicates("arm").set_index("arm")
rmse = main[main["metric"] == "rmse"].set_index("arm").loc[ARMS]
colors = {"none": "#999999", "mult": "#1b9e77", "add": "#d95f02"}
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(rmse.index, rmse["mean_12regions"],
       yerr=rmse["std_ddof0_12regions"], capsize=3,
       color=[colors[meta.loc[a, "form"]] for a in rmse.index])
ax.set_ylabel("RMSE (MW) - 12-region mean ± std")
ax.set_title("AU static-baseline arm RMSE (grey=uncorrected | green=multiplicative | orange=additive; UniAdd* is identical to Uni)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()

## 2. Mechanism Direction: Multiplicative vs. Additive on the GPM Base (UK Replication Check)

The UK static-baseline experiments observed **multiplicative dominance**. The direction
boolean fields below are generated by numeric rules (`results_summary.json.direction_checks`,
nothing is hand-typed).

In [ ]:
checks = summary["direction_checks"]
rows = []
for m in ["rmse", "mae", "corr"]:
    for sig, d in checks["per_metric"][m]["per_signal"].items():
        rows.append({"metric": m, "signal": sig,
                     "GPMpost (multiplicative)": d["gpm_mult_mean"],
                     "GPMadd (additive)": d["gpm_add_mean"],
                     "multiplicative dominates": d["gpm_mult_beats_additive"],
                     "multiplicative beats base": d["gpm_mult_improves_base"],
                     "additive beats base": d["gpm_add_improves_base"]})
with pd.option_context("display.float_format", "{:,.3f}".format):
    display(pd.DataFrame(rows))
print("Replication of the UK finding that multiplicative correction dominates on static bases (rmse, AND across all signals):",
      checks["uk_s6_static_mult_dominance_replicated_rmse"])
for m in ["rmse", "mae", "corr"]:
    print(f"  {m}: multiplicative dominance across all signals = "
          f"{checks['per_metric'][m]['static_mult_dominance_all_signals']}")

## 3. Degeneration Verification and Conservation Assertions

In [ ]:
deg = summary["uniform_additive_degeneration"]
print("-- Uniform x additive degeneration (alpha=0 => identical to baseline) --")
print("  Max within-group ptp (strict premise, should be exactly 0):",
      deg["uniform_base_within_group_ptp_max"])
print(f"  Max agent-level demand deviation (by signal):",
      {k: f"{v:.3e}" for k, v in deg["max_agent_demand_abs_dev_by_signal"].items()})
print(f"  Max metric-level deviation = {deg['metric_level_max_abs_dev_overall']:.3e} "
      f"(tolerance {deg['tol']:.0e}) -> all within tolerance = {deg['all_within_tol']}")

cons = summary["conservation"]
print("\n-- Per-SA3 demand conservation (asserted per SA3 for every (region, arm) demand array) --")
print(f"  Number of assertions = {cons['n_checks']}, max deviation = "
      f"{cons['per_sa3_max_abs_dev_mw']:.3e} MW (tolerance {cons['tol_mw']:.0e})"
      f"-> passed = {cons['passed']}")

## 4. PV Sensitivity Sub-Table (station_set = 'excl_pv')

Excludes 5 stations flagged by their day/night ratio -- **a pure statistical-layer
station-set filter**: allocation, factors, and the Voronoi assignment are not
re-run; the rows for these 5 stations are simply dropped during evaluation.
Unaffected regions are identical between the two tables.

In [ ]:
pv = summary["pv_sensitivity"]
print("Excluded stations:", pv["excluded_stations"])
print("Affected regions:", pv["affected_loc_keys"])

# Main/sub-table RMSE means side by side + per-region delta for affected regions (RMSE)
cmp_rows = []
for arm in ARMS:
    r_all = matrix[(matrix["arm"] == arm) & (matrix["metric"] == "rmse")
                   & (matrix["station_set"] == "all")].iloc[0]
    r_ex = matrix[(matrix["arm"] == arm) & (matrix["metric"] == "rmse")
                  & (matrix["station_set"] == "excl_pv")].iloc[0]
    cmp_rows.append({"arm": arm,
                     "RMSE main table": r_all["mean_12regions"],
                     "RMSE sub-table": r_ex["mean_12regions"],
                     "Delta (sub-main)": r_ex["mean_12regions"] - r_all["mean_12regions"]})
with pd.option_context("display.float_format", "{:,.3f}".format):
    display(pd.DataFrame(cmp_rows).set_index("arm"))

aff = pv["affected_loc_keys"]
diff = (matrix[matrix["station_set"] == "excl_pv"].set_index(["arm", "metric"])[aff]
        - matrix[matrix["station_set"] == "all"].set_index(["arm", "metric"])[aff])
print("Max absolute per-(arm, metric) delta across affected regions:")
display(diff.abs().max().rename("max |Delta|").to_frame())

---
*Generated 2026-07-14. Verified by `tests/test_b3.py` (16 checks), `data/processed/static/au_static_metrics.csv` / `results_summary.json`.*